In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd


df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "jrobischon/wikipedia-movie-plots",
    "wiki_movie_plots_deduped.csv",
)
# carrega o datset com as sinopses dos filmmes


In [2]:
df.shape
# tamanho do dataset

(34886, 8)

In [3]:
df.head()
#primeiras linhas

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [4]:
df.columns

Index(['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast',
       'Genre', 'Wiki Page', 'Plot'],
      dtype='object')

In [5]:
df['Plot'].iloc[0]
# pega uma sinopse da base

"A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]"

In [6]:
df_min = df.head(20).copy()
# copia as 20 primeiras linhas da base
# variavel menor
print(df_min[["Title", "Genre","Plot"]])
# mostra as tableas

                                                Title  \
0                              Kansas Saloon Smashers   
1                       Love by the Light of the Moon   
2                             The Martyred Presidents   
3                    Terrible Teddy, the Grizzly King   
4                              Jack and the Beanstalk   
5                                 Alice in Wonderland   
6                             The Great Train Robbery   
7                                     The Suburbanite   
8                            The Little Train Robbery   
9                          The Night Before Christmas   
10                           Dream of a Rarebit Fiend   
11  From Leadville to Aspen: A Hold-Up in the Rockies   
12                                Kathleen Mavourneen   
13                                       Daniel Boone   
14                    How Brown Saw the Baseball Game   
15                                       Laughing Gas   
16                           Th

In [7]:
import spacy

In [8]:
nlp = spacy.load("en_core_web_sm")

In [9]:
doc_teste = nlp(df_min["Plot"].iloc[0])
# roda o matcher na primeira siinopse
print(doc_teste)

A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]


In [10]:
for ent in doc_teste.ents:
    # vendo se o NER ja reconhece as entidades
    print(ent.text, ent.label_)

Irish NORP
Carrie Nation ORG
Irish NORP
Nation ORG


In [11]:
from spacy.matcher import Matcher

In [12]:
matcher = Matcher(nlp.vocab)
# atribui ao matcher o vocabulario do modelo

In [13]:
pattern1 = [{"POS": "PROPN", "OP":"+"}, {"POS": "VERB"}]
# atribui um dicionario comm a regra que procura um ou mais substantivos prróprios seguidos, para justamente pegar os nomes seguidos quee aparecem na sinpose
# e depois um verbo
# nisso pega a o maior token
# o objetivo desta regra é enconttrar o que o personagem faz baseado na sinopse

pattern2 = [{"POS": "PROPN"},{"POS": "VERB"}]
# adicionei uma regra para cobrir pronome e depois verbo

matcher.add("PERSONAGEM_ACAO",[pattern1,pattern2], greedy="LONGEST")
# adiciona ao matcher a regra

In [14]:
matches = matcher(doc_teste)
# roda o matcher no doc com as 20 sinopses
matches.sort(key=lambda x: x[1])
# ordena pelo primeiro token que o matcher acahar, dado a regra anterior

In [15]:
matches

[]

In [16]:
for i in range(len(df_min)):
    # para cada linha da base reduzida, transforama o texti em doc e aciona o matcher para procurar baseado na regra
    texto = df_min["Plot"].iloc[i]
    doc = nlp(texto)
    matches = matcher(doc)
    matches.sort(key=lambda x: x[1])
    
    if len(matches) > 0:
        # se tiver mais que um match mostra ele
        print(f"{df_min['Title'].iloc[i]}")
        for match_id, start, end in matches[:5]:# mostra as matches, seu token de inicio e token final
            print(" ", doc[start:end].text)
        print()

Jack and the Beanstalk
  Jack trading
  beig forced
  Jack wakes
  Jack celebrates

Alice in Wonderland
  Alice follows
  Queen invites

The Night Before Christmas
  Santa Claus leaves

Dream of a Rarebit Fiend
  Rarebit Fiend
  Fiend floats

Kathleen Mavourneen
  Terence O'More saves
  Charles Musser writes

Daniel Boone
  Indians encounter
  Boone has

How Brown Saw the Baseball Game
  Mr. Brown drinks



In [17]:
resumos = {}

for i in range(len(df_min)):
    texto = df_min["Plot"].iloc[i]
    # sinopse do filme
    titulo = df_min["Title"].iloc[i]
    # titulo
    doc = nlp(texto)
    # transforma em doc
    matches = matcher(doc)
    # acona o matcher
    matches.sort(key=lambda x: x[1])
    
    acoes = [doc[start:end].text for match_id, start, end in matches]
    # pega todas as ações que foram encontradas pelo matcher 
    resumos[titulo] = acoes
    # para o titulo do filme adiciona suas ações para cada um

for titulo, acoes in resumos.items():
    # para cada sinopse printa o titulo e suas acoes
    print(f"{titulo}: {', '.join(acoes)}\n")

Kansas Saloon Smashers: 

Love by the Light of the Moon: 

The Martyred Presidents: 

Terrible Teddy, the Grizzly King: 

Jack and the Beanstalk: Jack trading, beig forced, Jack wakes, Jack celebrates

Alice in Wonderland: Alice follows, Queen invites

The Great Train Robbery: 

The Suburbanite: 

The Little Train Robbery: 

The Night Before Christmas: Santa Claus leaves

Dream of a Rarebit Fiend: Rarebit Fiend, Fiend floats

From Leadville to Aspen: A Hold-Up in the Rockies: 

Kathleen Mavourneen: Terence O'More saves, Charles Musser writes

Daniel Boone: Indians encounter, Boone has

How Brown Saw the Baseball Game: Mr. Brown drinks

Laughing Gas: 

The Adventures of Dollie: 

The Black Viper: 

A Calamitous Elopement: 

The Call of the Wild: 



In [18]:
contagens = []
for i in range(len(df_min)):
    texto = df_min["Plot"].iloc[i]
    doc = nlp(texto)
    matches = matcher(doc)
    contagens.append(len(matches))

# mesmo loop para usar o matcher, porem agora para achar em 20 sinopses, qunatas nao foram achadas baseado na regra atual
df_min["QtdAcoes"] = contagens
print(f"Filmes sem nenhuma ação encontrada: {(df_min['QtdAcoes']==0).sum()} de {len(df_min)}")

Filmes sem nenhuma ação encontrada: 13 de 20


In [ ]:
doc_a = nlp(df_min["Plot"].iloc[0])
doc_b = nlp(df_min["Plot"].iloc[1])

print(df_min["Title"].iloc[0], ".", df_min["Title"].iloc[1])
print(doc_a.similarity(doc_b))
# compara a semelhança dos titulos

Kansas Saloon Smashers . Love by the Light of the Moon
0.8130687252054473


/tmp/ipykernel_108305/4208693692.py:5: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(doc_a.similarity(doc_b))
